<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/sessions/Session14_hugging_face_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۴: مدل‌های ازپیش‌آموزش‌دیده با Hugging Face: تحلیل احساس متن فارسی با ParsBERT

### اهداف این جلسه

- آشنایی با اکوسیستم Hugging Face: Hub، کتابخانه‌های `transformers` و `datasets`
- استفاده سریع از مدل‌ها با `pipeline`
- درک Tokenizerهای زیرکلمه‌ای (WordPiece)، `input_ids`، `attention_mask` و Tokenهای ویژه
- آماده‌سازی دیتاست با `datasets.map` و `DataCollatorWithPadding`
- Fine-tuning مدل ParsBERT برای تحلیل احساس نظرات فارسی با یک حلقه آموزش PyTorch
- ارزیابی با Macro-F1، ذخیره مدل و استفاده از آن روی جمله‌های جدید

> **نکتهٔ اجرا:** این نوت‌بوک روی GPU بسیار سریع‌تر اجرا می‌شود. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. پیش‌آموزش و Fine-tuning در NLP

در جلسه ۹ دیدیم که در بینایی ماشین، مدل آموزش‌دیده روی ImageNet نقطه شروع بسیار خوبی برای مسائل دیگر است. در پردازش زبان طبیعی، همین ایده با **BERT** (Devlin et al., 2019) فراگیر شد:

۱. **پیش‌آموزش (Pre-training):** یک Transformer Encoder (جلسه ۱۳) روی حجم عظیمی از متن **بدون برچسب** آموزش می‌بیند. در BERT وظیفه اصلی **Masked Language Modeling** است: ۱۵٪ Tokenها پنهان می‌شوند و مدل باید آن‌ها را از روی زمینه دوطرفه حدس بزند. مدل در این فرایند دانش زبانی گسترده‌ای (نحو، معنا، دانش عمومی) کسب می‌کند.

۲. **Fine-tuning:** یک لایه کوچک (مثلاً طبقه‌بند) روی مدل اضافه می‌شود و کل مدل با داده برچسب‌دار **کم** برای مسئله خاص آموزش می‌بیند.

برای فارسی، **ParsBERT** (Farahani et al., 2021) از گروه هوشواره روی میلیون‌ها سند فارسی پیش‌آموزش دیده است.

**Hugging Face** پرکاربردترین اکوسیستم برای این کار است:

| جزء | کاربرد |
|---|---|
| **Hub** | مخزن صدها هزار مدل و دیتاست؛ مثل گیت‌هاب برای مدل‌ها |
| `transformers` | بارگذاری و استفاده از مدل‌ها و Tokenizerها با رابط یکسان (`AutoModel`، `AutoTokenizer`) |
| `datasets` | بارگذاری و پردازش کارای دیتاست‌ها |

</div>

In [ ]:
%pip install -q transformers datasets

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup, pipeline)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
torch.manual_seed(42)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. شروع سریع با `pipeline`

`pipeline` ساده‌ترین راه استفاده از یک مدل است: پیش‌پردازش، اجرای مدل و پس‌پردازش را یکجا انجام می‌دهد. مدل ParsBERT پیش‌آموزش‌دیده را در وظیفه‌ای که با آن آموزش دیده، یعنی **پر کردن جای خالی (Fill-Mask)**، امتحان می‌کنیم:

</div>

In [ ]:
MODEL_NAME = 'HooshvareLab/bert-fa-base-uncased'
fill_mask = pipeline('fill-mask', model=MODEL_NAME, device=0 if device.type == 'cuda' else -1)

for sentence in ['تهران [MASK] ایران است.', 'امروز هوا خیلی [MASK] است.']:
    preds = fill_mask(sentence, top_k=5)
    print(sentence, '->', '، '.join(f"{p['token_str']} ({p['score']:.2f})" for p in preds))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل بدون هیچ آموزش اضافه‌ای، کلمات مناسب را از روی زمینه پیش‌بینی می‌کند. این دانش زبانی همان چیزی است که در Fine-tuning از آن استفاده می‌کنیم.

## ۳. Tokenizer

در جلسه ۱۱ Tokenizer ساده در سطح کلمه ساختیم و دیدیم که کلمات خارج از واژگان به `<unk>` تبدیل می‌شوند. مدل‌های مدرن از **Tokenizerهای زیرکلمه‌ای (Subword)** استفاده می‌کنند: کلمات پرتکرار یک Token هستند و کلمات کم‌تکرار به قطعات کوچک‌تر شکسته می‌شوند. BERT از **WordPiece** استفاده می‌کند (`##` یعنی ادامه کلمه قبلی)؛ GPT از **BPE**. این روش هم واژگان را محدود نگه می‌دارد (ParsBERT حدود ۱۰۰ هزار Token) و هم تقریباً هیچ کلمه‌ای ناشناخته نمی‌ماند؛ برای زبانی مثل فارسی با تکواژهای فراوان (می‌خورم، نخوردیم، خوردنی‌ها) بسیار مفید است.

</div>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('vocabulary size:', tokenizer.vocab_size)

text = 'کیفیت گوشی عالی بود ولی ارسالش خیلی دیر انجام شد'
print('tokens   :', tokenizer.tokenize(text))
enc = tokenizer(text)
print('input_ids:', enc['input_ids'])
print('decoded  :', tokenizer.decode(enc['input_ids']))
print('rare word:', tokenizer.tokenize('ناامیدکننده‌ترین'))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

خروجی Tokenizer سه نکته دارد:

- **Tokenهای ویژه:** `[CLS]` در ابتدای دنباله (بردار خروجی آن نماینده کل جمله برای طبقه‌بندی است) و `[SEP]` در انتها (و بین دو جمله در وظایف دوجمله‌ای).
- **`attention_mask`:** برای Batchهایی با طول متفاوت، مکان‌های Padding را مشخص می‌کند تا Attention به آن‌ها توجه نکند (همان ماسک جلسه ۱۲).
- **`truncation` و `max_length`:** BERT حداکثر ۵۱۲ Token می‌پذیرد؛ برای سرعت معمولاً طول کوتاه‌تری انتخاب می‌کنیم.

</div>

In [ ]:
batch = tokenizer(['خیلی خوب بود', 'اصلا راضی نبودم و پولم هدر رفت'], padding=True, truncation=True, max_length=16, return_tensors='pt')
for k, v in batch.items():
    print(f'{k:15s}', v.tolist())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. دیتاست: نظرات دیجی‌کالا و اسنپ‌فود

دیتاست `hezarai/sentiment-dksf` (پروژه متن‌باز هزار) شامل حدود ۳۱ هزار نظر کاربران فارسی‌زبان دیجی‌کالا و اسنپ‌فود با سه برچسب منفی، مثبت و خنثی است.

</div>

In [ ]:
raw = load_dataset('hezarai/sentiment-dksf')
print(raw)
LABELS = ['negative', 'positive', 'neutral']          # label ids 0, 1, 2 as defined by the dataset authors
id2label = dict(enumerate(LABELS))
label2id = {v: k for k, v in id2label.items()}

train_labels = np.array(raw['train']['label'])
print('\nlabel distribution (train):', {LABELS[i]: int((train_labels == i).sum()) for i in range(len(LABELS))})
for i in range(len(LABELS)):                         # sanity check: read a few examples of every label
    examples = raw['train'].filter(lambda ex: ex['label'] == i).select(range(2))['text']
    print(f'--- {LABELS[i]}:', ' | '.join(t[:80] for t in examples))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**همیشه چند نمونه از هر برچسب را بخوانید** تا مطمئن شوید نگاشت عدد به نام برچسب درست است؛ یک اشتباه در این مرحله کل ارزیابی را بی‌معنا می‌کند.

دیتاست فقط بخش‌های Train و Test دارد؛ ۱۰٪ داده آموزشی را برای اعتبارسنجی جدا می‌کنیم. سپس همه متن‌ها را با `map` به‌صورت دسته‌ای (Batched) Tokenize می‌کنیم. Padding را اینجا انجام نمی‌دهیم: `DataCollatorWithPadding` هر Batch را فقط تا طول بلندترین نمونه همان Batch پر می‌کند (**Dynamic Padding**) که بسیار کاراتر است.

</div>

In [ ]:
MAX_LEN = 128
split = raw['train'].train_test_split(test_size=0.1, seed=42)
datasets_ = {'train': split['train'], 'validation': split['test'], 'test': raw['test']}


def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)


tokenized = {name: ds.map(tokenize_batch, batched=True, remove_columns=['text']) for name, ds in datasets_.items()}
for name, ds in tokenized.items():
    ds.set_format('torch')
    print(f'{name:10s} {len(ds):6,} examples | columns: {ds.column_names}')

collator = DataCollatorWithPadding(tokenizer)
BATCH_SIZE = 32
train_loader = DataLoader(tokenized['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(tokenized['validation'], batch_size=128, collate_fn=collator)
test_loader = DataLoader(tokenized['test'], batch_size=128, collate_fn=collator)

b = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in b.items()})

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. مدل: ParsBERT با سر طبقه‌بندی

`AutoModelForSequenceClassification` بدنه ازپیش‌آموزش‌دیده BERT را بارگذاری می‌کند و یک لایه طبقه‌بند **جدید با وزن‌های تصادفی** روی بردار `[CLS]` اضافه می‌کند. هشدار «some weights were newly initialized» دقیقاً به همین لایه جدید اشاره دارد و طبیعی است. وقتی برچسب‌ها (`labels`) را به مدل بدهیم، خودش Cross-Entropy را حساب می‌کند و در `outputs.loss` برمی‌گرداند.

</div>

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params / 1e6:.1f} M')
print('classification head:', model.classifier)

with torch.no_grad():
    out = model(**{k: v.to(device) for k, v in b.items()})
print('logits:', tuple(out.logits.shape), '| initial loss:', round(out.loss.item(), 3), f'(~ ln 3 = {np.log(3):.3f})')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. Fine-tuning

تنظیمات استاندارد Fine-tuning مدل‌های BERT (پیشنهاد مقاله اصلی):

- **AdamW** با نرخ یادگیری کوچک، بین $2\times10^{-5}$ و $5\times10^{-5}$. نرخ بزرگ‌تر دانش پیش‌آموزش را از بین می‌برد (**Catastrophic Forgetting**).
- **Linear Warmup + Linear Decay** (جلسه ۵) با حدود ۶ تا ۱۰ درصد گام‌ها برای Warmup.
- فقط ۲ تا ۴ Epoch؛ مدل خیلی سریع روی داده کم Overfit می‌شود.
- Weight Decay روی وزن‌ها، اما **نه** روی بایاس‌ها و پارامترهای LayerNorm.
- Mixed Precision (جلسه ۹) برای سرعت روی GPU.

</div>

In [ ]:
EPOCHS = 2
LR = 3e-5
USE_AMP = device.type == 'cuda'

no_decay = ('bias', 'LayerNorm.weight')
param_groups = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
optimizer = torch.optim.AdamW(param_groups, lr=LR)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.06 * total_steps), num_training_steps=total_steps)
scaler = torch.amp.GradScaler(enabled=USE_AMP)


@torch.no_grad()
def predict(loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**{k: v for k, v in batch.items() if k != 'labels'}).logits
        preds.append(logits.argmax(-1).cpu()); labels.append(batch['labels'].cpu())
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


for epoch in range(1, EPOCHS + 1):
    model.train()
    start, running = time.time(), 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            loss = model(**batch).loss
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                               # unscale before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
        if step % 200 == 0:
            print(f'  epoch {epoch} step {step}/{len(train_loader)} | loss {running / 200:.4f}')
            running = 0.0
    p, y = predict(val_loader)
    print(f'epoch {epoch} | val accuracy {accuracy_score(y, p):.4f} | val macro-F1 {f1_score(y, p, average="macro"):.4f} | {time.time() - start:.0f}s')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۷. ارزیابی روی داده آزمون

کلاس‌ها نامتوازن‌اند؛ پس علاوه بر Accuracy، **Macro-F1** (میانگین F1 کلاس‌ها با وزن برابر) را گزارش می‌کنیم تا عملکرد ضعیف روی کلاس کم‌تعداد پنهان نماند. معیار استاندارد مقالات تحلیل احساس هم معمولاً همین است.

</div>

In [ ]:
p, y = predict(test_loader)
print(f'test accuracy: {accuracy_score(y, p):.4f} | test macro-F1: {f1_score(y, p, average="macro"):.4f}\n')
print(classification_report(y, p, target_names=LABELS, digits=3))
ConfusionMatrixDisplay(confusion_matrix(y, p), display_labels=LABELS).plot(cmap='Blues', colorbar=False)
plt.title('ParsBERT on sentiment-dksf (test)'); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

همان‌طور که انتظار می‌رود، کلاس **خنثی** سخت‌ترین است: مرز آن با دو کلاس دیگر مبهم است و حتی برچسب‌زن‌های انسانی هم در آن اختلاف نظر دارند.

## ۸. استفاده از مدل و ذخیره آن

مدل Fine-tune‌شده را می‌توانیم مستقیماً در یک `pipeline` قرار دهیم. چون `id2label` را تنظیم کرده‌ایم، خروجی نام برچسب‌ها را دارد. با `save_pretrained` مدل و Tokenizer در یک پوشه ذخیره می‌شوند و با `push_to_hub` می‌توان آن را در Hugging Face Hub منتشر کرد.

</div>

In [ ]:
classifier = pipeline('text-classification', model=model, tokenizer=tokenizer, device=0 if device.type == 'cuda' else -1)
samples = [
    'غذا گرم و خوشمزه بود و پیک خیلی سریع رسید',
    'کیفیت پارچه افتضاح بود و اصلا شبیه عکس نبود',
    'گوشی رو دیروز تحویل گرفتم، هنوز کامل امتحانش نکردم',
    'قیمتش نسبت به کیفیتش منطقیه ولی بسته‌بندی ضعیف بود',
]
for s, r in zip(samples, classifier(samples)):
    print(f"{r['label']:9s} ({r['score']:.2f})  {s}")

model.save_pretrained('parsbert-sentiment-dksf')
tokenizer.save_pretrained('parsbert-sentiment-dksf')
# model.push_to_hub('your-username/parsbert-sentiment-dksf')   # requires `huggingface-cli login`

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### روش جایگزین: `Trainer`

کتابخانه `transformers` کلاس `Trainer` را دارد که حلقه آموزش بالا (به‌علاوه Mixed Precision، ارزیابی دوره‌ای، ذخیره Checkpoint و گزارش‌دهی) را در چند خط انجام می‌دهد. ما حلقه را دستی نوشتیم تا دقیقاً بدانیم چه اتفاقی می‌افتد؛ در پروژه‌های واقعی `Trainer` (یا کتابخانه‌هایی مثل Lightning و Accelerate) کار را ساده‌تر می‌کند:

```python
from transformers import Trainer, TrainingArguments

args = TrainingArguments(output_dir='out', learning_rate=3e-5, num_train_epochs=2,
                         per_device_train_batch_size=32, warmup_ratio=0.06, weight_decay=0.01,
                         fp16=True, eval_strategy='epoch')
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'],
                  eval_dataset=tokenized['validation'], data_collator=collator)
trainer.train()
```

## جمع‌بندی

- الگوی غالب NLP امروز: پیش‌آموزش روی متن عظیم بدون برچسب، سپس Fine-tuning با داده برچسب‌دار کم.
- `pipeline` برای استفاده سریع؛ `AutoTokenizer` و `AutoModelFor...` برای کنترل کامل.
- Tokenizerهای زیرکلمه‌ای (WordPiece، BPE) مشکل کلمات ناشناخته را حل می‌کنند؛ خروجی آن‌ها `input_ids` و `attention_mask` است.
- `datasets.map(batched=True)` برای Tokenize کردن و `DataCollatorWithPadding` برای Padding پویا.
- Fine-tuning: AdamW با نرخ یادگیری حدود $3\times10^{-5}$، Warmup خطی، ۲ تا ۴ Epoch.
- برای داده نامتوازن Macro-F1 گزارش دهید؛ نگاشت برچسب‌ها را با خواندن نمونه‌ها بررسی کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. نرخ یادگیری را $10^{-5}$ و $10^{-4}$ بگیرید و Macro-F1 اعتبارسنجی را مقایسه کنید. با نرخ بزرگ چه اتفاقی می‌افتد؟

۲. مدل چندزبانه `xlm-roberta-base` را به‌جای ParsBERT Fine-tune کنید و نتیجه را مقایسه کنید.

۳. فقط سر طبقه‌بند را آموزش دهید و بدنه BERT را منجمد کنید (مثل Feature Extraction در جلسه ۹). چقدر افت دقت دارد؟

۴. با `MAX_LEN = 64` آموزش دهید. زمان آموزش و دقت چه تغییری می‌کند؟ با هیستوگرام طول نظرات (بر حسب Token) تصمیم بگیرید چه طولی مناسب است.

۵. مدل آموزش‌دیده را با `push_to_hub` در حساب Hugging Face خودتان منتشر کنید و یک Model Card برای آن بنویسید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** نرخ یادگیری را $10^{-5}$ و $10^{-4}$ بگیرید و Macro-F1 اعتبارسنجی را مقایسه کنید. با نرخ بزرگ چه اتفاقی می‌افتد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** مدل چندزبانه `xlm-roberta-base` را به‌جای ParsBERT Fine-tune کنید و نتیجه را مقایسه کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** فقط سر طبقه‌بند را آموزش دهید و بدنه BERT را منجمد کنید (مثل Feature Extraction در جلسه ۹). چقدر افت دقت دارد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** با `MAX_LEN = 64` آموزش دهید. زمان آموزش و دقت چه تغییری می‌کند؟ با هیستوگرام طول نظرات (بر حسب Token) تصمیم بگیرید چه طولی مناسب است.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۵.** مدل آموزش‌دیده را با `push_to_hub` در حساب Hugging Face خودتان منتشر کنید و یک Model Card برای آن بنویسید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید
